# Fase 6 — PCA + Clustering de Clientes

**Proyecto Final — Gestión de Datos (UAX) · 3º Ingeniería Matemática**  
**Paleta:** Deep Sky — azul oceano + coral + dorado  

Esta fase segmenta automáticamente los 5.750 clientes usando Machine Learning, partiendo de las métricas CLTV calculadas en la Fase 5. Es la sección de mayor peso analítico del proyecto.

```
cltv_resultados.csv
       │
       ▼
  Pairplot exploratorio  ──►  Decisiones informadas
       │
       ▼
  Selección features  ──►  Outliers IQR×3  ──►  StandardScaler
       │
       ▼
  PCA (varianza + biplot + scatter)  ──►  2D para visualización
       │
       ▼
  Elbow + Silhouette  ──►  K óptimo
       │
       ▼
  K-Means (5D)  ──►  Perfil clusters  ──►  Etiquetas negocio
       │
       ▼
  clustering_resultados.csv
```

---

## Tabla de contenidos

| # | Sección |
|---|---|
| 1 | Imports, paleta y carga de datos |
| 2 | Pairplot exploratorio → decisiones |
| 3 | Matriz de correlación |
| 4 | Preparación: outliers + normalización |
| 5 | PCA: varianza, biplot, scatter |
| 6 | Selección K: Elbow + Silhouette |
| 7 | K-Means clustering |
| 8 | Análisis profundo de clusters |
| 9 | Radar chart por cluster |
| 10 | Interpretación y etiquetas de negocio |
| 11 | Comparativa CLTV vs Cluster |
| 12 | Exportación y KPIs finales |

---
## 1. Imports, paleta y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# ═══════════════════════════════════════════════════════════════
# PALETA  "DEEP SKY"  —  Fase 6  (PCA + Clustering)
# Progresión del proyecto: Azul (EDA) → Verde (ETL) → Ámbar (CLTV) → Deep Sky
# ═══════════════════════════════════════════════════════════════
DS1 = '#03045E'   # azul marino profundo   — títulos principales
DS2 = '#0077B6'   # azul océano            — barras primarias
DS3 = '#00B4D8'   # cian vibrante          — acento principal
DS4 = '#90E0EF'   # cian claro             — barras secundarias / fondos
DS5 = '#FF6B6B'   # coral-rojo             — contraste / outliers
DS6 = '#FFD93D'   # ámbar dorado           — highlights / KPIs
DS7 = '#6BCB77'   # verde menta            — positivo / alto valor
DS8 = '#7F8C8D'   # gris neutro            — anotaciones

# Colores de clusters (máxima diferenciación visual, 7 disponibles)
CLUSTER_PAL = ['#FF6B6B','#4D96FF','#6BCB77','#FFD93D','#A29BFE','#FF9F43','#00B4D8']

SEQ_CMAP  = 'YlGnBu'    # colormap secuencial para heatmaps
DIV_CMAP  = 'RdYlBu_r'  # colormap divergente

# Estilo general
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#F8FAFC',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
})

GRAFICOS = Path('graficos')
GRAFICOS.mkdir(exist_ok=True)

# Barra visual de la paleta
colores_pal = [DS1,DS2,DS3,DS4,DS5,DS6,DS7]
nombres_pal = ['DS1\nazul noche','DS2\nocéano','DS3\ncian','DS4\ncian claro','DS5\ncoral','DS6\ndorado','DS7\nverde']
fig, ax = plt.subplots(figsize=(13, 1.1))
for i,(c,n) in enumerate(zip(colores_pal, nombres_pal)):
    ax.barh(0, 1, left=i, color=c, edgecolor='white', linewidth=2)
    ax.text(i+0.5, 0, n, ha='center', va='center', fontsize=8,
            color='white' if i<3 else ('black' if i==3 else 'white'))
ax.set_xlim(0,7); ax.axis('off')
ax.set_title('Paleta Deep Sky — Fase 6 PCA + Clustering', fontsize=11, pad=5)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_00_paleta.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close()
print('Paleta cargada.')

In [ ]:
df = pd.read_csv('cltv_resultados.csv')

# Columnas mínimas requeridas
REQUERIDAS = ['customer_id','ingresos_t','margen_t','frecuencia_t',
              'ticket_medio','tasa_devolucion','cltv','segmento_cltv']
faltantes = [c for c in REQUERIDAS if c not in df.columns]
if faltantes:
    raise RuntimeError(f'Faltan columnas: {faltantes} — ejecuta primero 04_cltv.ipynb')

print(f'Filas cargadas : {len(df):,}')
print(f'Columnas       : {len(df.columns)}')
print(f'Segmentos CLTV : {dict(df["segmento_cltv"].value_counts())}')
print()
display(df[REQUERIDAS].describe().round(3))

---
## 2. Pairplot exploratorio → decisiones

Antes de aplicar ningún algoritmo, visualizamos **todas las combinaciones de features** coloreadas por segmento CLTV (Alto/Medio/Bajo). Este gráfico guía todas las decisiones posteriores:
- ¿Qué variables separan mejor los grupos?
- ¿Hay variables redundantes (muy correlacionadas)?
- ¿Hay outliers extremos?
- ¿Los datos tienen estructura natural para el clustering?

> Puede tardar ~2 minutos.

In [ ]:
# ───────────────────────────────────────────────────────────────────
# FEATURE SET DEL CLUSTERING
# ───────────────────────────────────────────────────────────────────
# El notebook puede trabajar con dos escenarios:
#   1. CSV enriquecido con features de diversidad (post 04b_features_diversidad.ipynb)
#   2. CSV estándar de 04_cltv.ipynb, derivando aquí las features logarítmicas básicas
#
# Objetivo: evitar que la Fase 6 dependa obligatoriamente de 04b para poder ejecutarse.

if 'log_ingresos' not in df.columns and 'ingresos_t' in df.columns:
    df['log_ingresos'] = np.log1p(df['ingresos_t'])
if 'log_ticket_medio' not in df.columns and 'ticket_medio' in df.columns:
    df['log_ticket_medio'] = np.log1p(df['ticket_medio'])

FEATURES_BASE = ['log_ingresos', 'num_ventas', 'log_ticket_medio',
                 'dias_sin_compra', 'customer_age_days', 'tasa_devolucion']
FEATURES_OPCIONALES = ['n_productos_distintos', 'n_tiendas_distintas',
                       'n_categorias_distintas', 'lineas_por_venta']

faltantes_base = [f for f in FEATURES_BASE if f not in df.columns]
if faltantes_base:
    raise RuntimeError(f'Faltan columnas base para clustering: {faltantes_base}\n'
                       f'→ Ejecuta primero 04_cltv.ipynb o revisa cltv_resultados.csv')

FEATURES_TODAS = FEATURES_BASE + [f for f in FEATURES_OPCIONALES if f in df.columns]

# Paleta por segmento CLTV
PAL_SEG = {'Alto': DS2, 'Medio': DS3, 'Bajo': DS5}

print('Features disponibles para exploración:')
for f in FEATURES_TODAS:
    print(f'  · {f}')
print()

print('Generando pairplot exploratorio del feature set disponible...')
pp = sns.pairplot(
    df[FEATURES_TODAS + ['segmento_cltv']].sample(min(3000, len(df)), random_state=42),
    hue='segmento_cltv', hue_order=['Alto','Medio','Bajo'],
    palette=PAL_SEG,
    plot_kws={'alpha':0.25, 's':6},
    diag_kind='kde',
    corner=False
)
pp.figure.suptitle(f'Pairplot exploratorio — feature set disponible ({len(FEATURES_TODAS)} features)',
                   y=1.005, fontsize=12, color=DS1, fontweight='bold')
plt.savefig(GRAFICOS/'pca_01_pairplot_exploratorio.png', dpi=90, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_01_pairplot_exploratorio.png')


In [ ]:
# Observaciones cuantitativas del pairplot:
# separación de grupos = coeficiente de variación intergrupo

print('=== ANÁLISIS CUANTITATIVO DE SEPARACIÓN POR FEATURE ===')
print()
resultados_sep = []
for feat in FEATURES_TODAS:
    medias = df.groupby('segmento_cltv')[feat].mean()
    media_global = df[feat].mean()
    std_global   = df[feat].std()
    # Rango entre segmentos relativo a la std global
    rango_rel = (medias.max() - medias.min()) / (std_global + 1e-9)
    resultados_sep.append({'feature': feat, 'separacion': round(rango_rel, 3),
                           'media_Alto': round(medias.get('Alto',0),3),
                           'media_Medio': round(medias.get('Medio',0),3),
                           'media_Bajo':  round(medias.get('Bajo',0),3)})

sep_df = pd.DataFrame(resultados_sep).sort_values('separacion', ascending=False)
display(sep_df)
print()

# Barplot de separación
fig, ax = plt.subplots(figsize=(10, 4))
colores_b = [DS2 if v >= sep_df['separacion'].quantile(0.5) else DS4
             for v in sep_df['separacion']]
bars = ax.barh(sep_df['feature'], sep_df['separacion'],
               color=colores_b, edgecolor='white', height=0.6)
for bar, v in zip(bars, sep_df['separacion']):
    ax.text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=10)
ax.set_title('Poder discriminante de cada feature\n(rango intergrupo / desviación típica global)',
             fontweight='bold')
ax.set_xlabel('Separación relativa (mayor = más discriminante)')
ax.axvline(1.0, color=DS5, linestyle='--', alpha=0.7, label='Umbral = 1')
ax.legend()
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_02_separacion_features.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_02_separacion_features.png')
print()
top3 = sep_df.head(3)['feature'].tolist()
print(f'Top 3 features más discriminantes: {top3}')
print('→ Estas serán las features más relevantes para el clustering.')

---
## 3. Matriz de correlación

Antes de seleccionar features, comprobamos qué variables son redundantes (correlación alta) para evitar dar doble peso a la misma información.

In [ ]:
# Matriz de correlación entre features enriquecidas
# Referencia: log_ingresos (eje monetario dominante tras los logs)
REF_FEAT = 'log_ingresos'

corr_matrix = df[FEATURES_TODAS].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap correlación
sns.heatmap(corr_matrix, ax=axes[0], annot=True, fmt='.2f',
            cmap=DIV_CMAP, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Correlación de Pearson'})
axes[0].set_title('Matriz de correlación — feature set enriquecido', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Barplot de correlación con la feature de referencia
corr_ref = corr_matrix[REF_FEAT].drop(REF_FEAT).sort_values(ascending=True)
colores_corr = [DS2 if v > 0 else DS5 for v in corr_ref]
bars = axes[1].barh(corr_ref.index, corr_ref.values,
                    color=colores_corr, edgecolor='white', height=0.6)
for bar, v in zip(bars, corr_ref.values):
    axes[1].text(v + (0.01 if v >= 0 else -0.01),
                 bar.get_y()+bar.get_height()/2,
                 f'{v:.3f}', va='center',
                 ha='left' if v >= 0 else 'right', fontsize=10)
axes[1].axvline(0, color=DS8, linewidth=1)
axes[1].set_title(f'Correlación de cada feature con {REF_FEAT}', fontweight='bold')
axes[1].set_xlabel('Correlación de Pearson')
axes[1].set_xlim(-1.1, 1.1)

plt.tight_layout()
plt.savefig(GRAFICOS/'pca_03_correlacion.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_03_correlacion.png')
print()

# Detectar pares con correlación alta (informativo)
print('Pares con |correlación| > 0.85 (posible redundancia, se gestionan en celda 10):')
encontrados = False
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i,j]
        if abs(r) > 0.85:
            print(f'  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: r={r:.3f}')
            encontrados = True
if not encontrados:
    print('  Ningún par supera |r|=0.85 — todas las features aportan información independiente.')


---
## 4. Preparación de features

Preparamos un **set final diversificado** que preserve dimensiones de negocio distintas:
1. **Monetario**
2. **Frecuencia / intensidad**
3. **Recency**
4. **Tenure**
5. **Postventa**

La selección no busca dejar "las menos correlacionadas posibles" de forma ciega, sino evitar que
toda la segmentación colapse en una sola familia de variables.

In [ ]:
# ───────────────────────────────────────────────────────────────────
# SELECCIÓN DE FEATURES — preservando dimensiones de negocio
# ───────────────────────────────────────────────────────────────────
# En vez de eliminar por correlación de forma ciega hasta quedarnos sin eje monetario,
# elegimos una representante por dimensión de negocio y añadimos extras solo si aportan señal.

sep_map = dict(zip(sep_df['feature'], sep_df['separacion'])) if 'sep_df' in locals() else {}

GRUPOS_FEATURES = {
    'monetario': ['log_ingresos', 'log_ticket_medio'],
    'frecuencia': ['num_ventas', 'n_productos_distintos', 'n_tiendas_distintas',
                   'n_categorias_distintas', 'lineas_por_venta'],
    'recency': ['dias_sin_compra'],
    'tenure': ['customer_age_days'],
    'postventa': ['tasa_devolucion'],
}

def elegir_representante(candidatas):
    disponibles = [f for f in candidatas if f in FEATURES_TODAS]
    if not disponibles:
        return None
    return sorted(
        disponibles,
        key=lambda f: (sep_map.get(f, 0), df[f].std()),
        reverse=True,
    )[0]

FEATURES = []
for grupo, candidatas in GRUPOS_FEATURES.items():
    elegida = elegir_representante(candidatas)
    if elegida is not None:
        FEATURES.append(elegida)

FEATURES = list(dict.fromkeys(FEATURES))

print('Features seleccionadas por dimensión:')
for grupo, candidatas in GRUPOS_FEATURES.items():
    disponibles = [f for f in candidatas if f in FEATURES_TODAS]
    elegida = next((f for f in FEATURES if f in disponibles), None)
    if elegida is not None:
        print(f'  {grupo:<10} → {elegida}  (sep={sep_map.get(elegida, np.nan):.3f}, std={df[elegida].std():.3f})')

corr_sel = df[FEATURES].corr().round(3)
print('\nCorrelación entre features seleccionadas:')
display(corr_sel)

X = df[FEATURES].copy()
print(f'Features finales ({len(FEATURES)}): {FEATURES}')
print(f'Nulos en X: {X.isnull().sum().sum()}')


In [ ]:
# Recorte de outliers extremos: IQR × 3
# Política del proyecto: nunca eliminar clientes → solo clipear
# Salvaguarda importante: si IQR = 0, NO se clipea la variable.
# De lo contrario podríamos colapsar toda su varianza a una constante.
X_clean = X.copy()
info_clip = []

for feat in FEATURES:
    q1  = X[feat].quantile(0.25)
    q3  = X[feat].quantile(0.75)
    iqr = q3 - q1

    if np.isclose(iqr, 0):
        lo = X[feat].min()
        hi = X[feat].max()
        n_clip = 0
        accion = 'sin clip (IQR=0)'
        X_clean[feat] = X[feat]
    else:
        lo  = q1 - 3*iqr
        hi  = q3 + 3*iqr
        n_clip = ((X[feat] < lo) | (X[feat] > hi)).sum()
        accion = 'clip IQR×3'
        X_clean[feat] = X[feat].clip(lo, hi)

    info_clip.append({'feature': feat, 'accion': accion, 'n_recortados': int(n_clip),
                      'lim_inferior': round(float(lo), 2), 'lim_superior': round(float(hi), 2),
                      'std_original': round(float(X[feat].std()), 4),
                      'std_final': round(float(X_clean[feat].std()), 4)})

clip_df = pd.DataFrame(info_clip)
print('Recorte outliers (IQR × 3):')
display(clip_df)
print(f'\nFilas conservadas: {len(X_clean):,} (0 eliminadas)')

# Eliminar únicamente features que realmente queden sin varianza
std_post = X_clean.std(ddof=0)
features_sin_var = std_post[std_post <= 1e-8].index.tolist()
if features_sin_var:
    print(f'\nSe eliminan features sin varianza tras preparar datos: {features_sin_var}')
    FEATURES = [f for f in FEATURES if f not in features_sin_var]
    X_clean = X_clean[FEATURES].copy()

# Normalización
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

print(f'\nFeatures efectivas para clustering ({len(FEATURES)}): {FEATURES}')
print(f'StandardScaler aplicado.')
print(f'  Media por feature (debe ~0): {np.round(X_scaled.mean(axis=0),4)}')
print(f'  Std  por feature (debe ~1): {np.round(X_scaled.std(axis=0),4)}')

In [ ]:
# Comparación antes/después de normalizar
fig, axes = plt.subplots(2, 5, figsize=(18, 8))

for i, feat in enumerate(FEATURES):
    # Fila superior: original
    axes[0,i].hist(X_clean[feat], bins=40, color=DS2, edgecolor='white', alpha=0.85, linewidth=0.4)
    axes[0,i].set_title(f'{feat}\n(original recortado)', fontsize=8)
    axes[0,i].tick_params(labelsize=7)

    # Fila inferior: normalizado
    axes[1,i].hist(X_scaled[:,i], bins=40, color=DS3, edgecolor='white', alpha=0.85, linewidth=0.4)
    axes[1,i].set_title(f'{feat}\n(StandardScaler)', fontsize=8)
    axes[1,i].tick_params(labelsize=7)

axes[0,0].set_ylabel('Original', fontsize=10)
axes[1,0].set_ylabel('Normalizado', fontsize=10)

plt.suptitle('Features antes y después de normalizar', fontsize=12,
             fontweight='bold', color=DS1, y=1.01)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_05_normalizacion.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_05_normalizacion.png')

---
## 5. PCA — Reducción de dimensionalidad

**Propósito dual:**
1. Entender qué combinación lineal de features explica más varianza (biplot de loadings)
2. Reducir a 2D para visualizar los clusters en un scatter plot

> El K-Means se aplica en el espacio normalizado **5D completo**. PCA solo sirve para visualización.

In [ ]:
# PCA completo para análisis de varianza
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

var_exp = pca_full.explained_variance_ratio_
var_cum = np.cumsum(var_exp)
n_comp  = len(var_exp)

print('Varianza explicada por componente:')
for i,(v,vc) in enumerate(zip(var_exp, var_cum), 1):
    barra = '█' * int(v*50)
    print(f'  PC{i}: {v*100:5.1f}%  acum={vc*100:.1f}%  {barra}')

n80 = next(i for i,vc in enumerate(var_cum,1) if vc>=0.80)
n90 = next(i for i,vc in enumerate(var_cum,1) if vc>=0.90)
print(f'\nComponentes para ≥80% varianza: {n80}')
print(f'Componentes para ≥90% varianza: {n90}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras varianza por componente
colores_pc = [DS2 if i<n80 else DS4 for i in range(n_comp)]
bars = axes[0].bar(range(1,n_comp+1), var_exp*100,
                   color=colores_pc, edgecolor='white', width=0.6)
for bar,v in zip(bars, var_exp*100):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{v:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].set_title('Varianza explicada por componente\n(azul oscuro = necesarios para ≥80%)',
                  fontweight='bold')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada (%)')
axes[0].set_xticks(range(1,n_comp+1))
p1 = mpatches.Patch(color=DS2, label=f'Necesarios para ≥80% (PC1..PC{n80})')
p2 = mpatches.Patch(color=DS4, label='Componentes adicionales')
axes[0].legend(handles=[p1,p2], fontsize=9)

# Varianza acumulada
axes[1].plot(range(1,n_comp+1), var_cum*100,
             marker='o', color=DS2, linewidth=2.5, markersize=8, zorder=3)
axes[1].fill_between(range(1,n_comp+1), var_cum*100, alpha=0.12, color=DS3)
axes[1].axhline(80, color=DS6, linestyle='--', linewidth=1.5, label='80%')
axes[1].axhline(90, color=DS5, linestyle='--', linewidth=1.5, label='90%')
for i,(v,vc) in enumerate(zip(var_exp, var_cum),1):
    axes[1].annotate(f'{vc*100:.0f}%', xy=(i, vc*100),
                     xytext=(7,4), textcoords='offset points',
                     fontsize=9, color=DS1)
axes[1].set_title('Varianza acumulada', fontweight='bold')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].set_xticks(range(1,n_comp+1))
axes[1].set_ylim(0,105)
axes[1].legend()

plt.suptitle('PCA — Análisis de varianza explicada', fontsize=13,
             fontweight='bold', color=DS1, y=1.01)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_06_varianza.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_06_varianza.png')

In [ ]:
# Reducción a 2D
pca2 = PCA(n_components=2, random_state=42)
X_pca = pca2.fit_transform(X_scaled)
df['pc1'] = X_pca[:,0]
df['pc2'] = X_pca[:,1]

var1 = pca2.explained_variance_ratio_[0]*100
var2 = pca2.explained_variance_ratio_[1]*100
print(f'PC1: {var1:.1f}% | PC2: {var2:.1f}% | Total: {var1+var2:.1f}%')

# Loadings
loadings = pca2.components_.T
loadings_df = pd.DataFrame(loadings, index=FEATURES, columns=['PC1','PC2']).round(3)
print(f'\nLoadings (contribución de cada feature):')
display(loadings_df)

In [ ]:
# Biplot sofisticado: fondo = clientes por segmento CLTV, flechas = loadings
# Adaptado al feature set enriquecido (10 features → paleta cíclica + ejes fijos)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Izquierda: Biplot con segmento CLTV ──────────────────────────────────
ax = axes[0]
for seg, color in PAL_SEG.items():
    mask = df['segmento_cltv'] == seg
    ax.scatter(df.loc[mask,'pc1'], df.loc[mask,'pc2'],
               s=10, alpha=0.3, color=color, label=f'{seg} ({mask.sum():,})', zorder=1)

# Calcular escala dinámica para que las flechas se vean
pc_max = max(abs(df['pc1']).max(), abs(df['pc2']).max())
load_max = np.abs(loadings).max()
scale = pc_max * 0.7 / (load_max + 1e-9)

# Paleta cíclica de colores para flechas (suficiente para 10+ features)
colores_fl_pool = [DS1, DS2, DS3, DS5, DS6, DS7, '#8E44AD', '#16A085', '#D35400', '#2C3E50']

for i, feat in enumerate(FEATURES):
    color = colores_fl_pool[i % len(colores_fl_pool)]
    lx, ly = loadings[i,0]*scale, loadings[i,1]*scale
    ax.annotate('', xy=(lx,ly), xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    # Offset proporcional para que no se solapen
    ox = 0.15 if lx>=0 else -0.15
    oy = 0.15 if ly>=0 else -0.15
    ax.text(lx+ox, ly+oy, feat, fontsize=9, color=color,
            fontweight='bold', ha='center')

ax.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.6)
ax.axvline(0, color='gray', lw=0.5, ls='--', alpha=0.6)
# FIJAR LÍMITES para que la imagen no se descontrole
margen = pc_max * 1.1
ax.set_xlim(-margen, margen)
ax.set_ylim(-margen, margen)
ax.set_aspect('equal', adjustable='box')

ax.set_title(f'Biplot PCA — fondo=segmento CLTV, flechas=loadings\n(PC1={var1:.1f}%, PC2={var2:.1f}%)',
             fontweight='bold')
ax.set_xlabel(f'PC1 ({var1:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({var2:.1f}% varianza)')
ax.legend(fontsize=9, title='Segmento CLTV', loc='best')

# ── Derecha: Heatmap de loadings (todas las componentes) ─────────────────
ax2 = axes[1]
loadings_full = pd.DataFrame(
    pca_full.components_[:n_comp].T,
    index=FEATURES,
    columns=[f'PC{i}' for i in range(1,n_comp+1)]
).round(3)
sns.heatmap(loadings_full, ax=ax2, annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Loading'})
ax2.set_title('Loadings completos\n(todas las componentes principales)',
              fontweight='bold')
ax2.set_xlabel('Componente principal')
ax2.set_ylabel('Feature')

plt.suptitle('PCA — Biplot y mapa de loadings', fontsize=13,
             fontweight='bold', color=DS1, y=1.01)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_07_biplot.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_07_biplot.png')

print()
print('Interpretación de los loadings:')
pc1_top = loadings_df['PC1'].abs().idxmax()
pc2_top = loadings_df['PC2'].abs().idxmax()
print(f'  PC1 dominada por: {pc1_top} (loading={loadings_df.loc[pc1_top,"PC1"]:+.3f})')
print(f'  PC2 dominada por: {pc2_top} (loading={loadings_df.loc[pc2_top,"PC2"]:+.3f})')


---
## 6. Selección del K óptimo: Elbow + Silhouette

Dos métodos complementarios que deben **coincidir** para dar confianza en el K elegido:

| Método | Mide | Criterio |
|---|---|---|
| **Elbow** | Inercia intra-cluster | Codo visible en la curva |
| **Silhouette** | Separación inter/intra cluster | Máximo valor |

Adicional: **Silhouette diagram** para el K óptimo — muestra la calidad de cada punto individual.

In [ ]:
from sklearn.metrics import davies_bouldin_score

K_RANGE  = range(2, 10)
inercias    = []
silhouettes = []
db_scores   = []

print('Calculando Elbow + Silhouette + Davies-Bouldin para K=2..9...')
print(f'  {"K":>3}  {"Inercia":>12}  {"Silhouette":>12}  {"Davies-Bouldin":>15}')
print('  ' + '-'*46)
for k in K_RANGE:
    km  = KMeans(n_clusters=k, random_state=42, n_init=15)
    lbl = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    sil = silhouette_score(X_scaled, lbl)
    db  = davies_bouldin_score(X_scaled, lbl)
    silhouettes.append(sil)
    db_scores.append(db)
    print(f'  {k:>3}  {km.inertia_:>12,.0f}  {sil:>12.4f}  {db:>15.4f}')

k_optimo_sil = list(K_RANGE)[silhouettes.index(max(silhouettes))]
k_optimo_db  = list(K_RANGE)[db_scores.index(min(db_scores))]

print()
print(f'K óptimo según Silhouette     (máximo): {k_optimo_sil}')
print(f'K óptimo según Davies-Bouldin (mínimo): {k_optimo_db}')

# ── Selección final basada en métricas (NO en restricción artificial) ────
# Con el feature set enriquecido el óptimo matemático suele caer en K=3-5,
# que coincide con un mínimo accionable de negocio. No imponemos K_MIN.

if k_optimo_sil == k_optimo_db:
    k_optimo = k_optimo_sil
    print(f'\n✅ Silhouette y Davies-Bouldin coinciden → K = {k_optimo}')
else:
    # Si discrepan, prevalece Silhouette (más robusto a forma del cluster)
    k_optimo = k_optimo_sil
    print(f'\n⚠️  Silhouette={k_optimo_sil} vs Davies-Bouldin={k_optimo_db}')
    print(f'    Se elige K={k_optimo} (criterio Silhouette, más robusto)')

# Tabla resumen para la memoria técnica
print()
print('=== TABLA DE DECISIÓN ===')
print(f'  {"K":>3}  {"Silhouette":>12}  {"Davies-B":>10}  {"Decisión":>12}')
print('  ' + '-'*42)
for k, s, d in zip(K_RANGE, silhouettes, db_scores):
    marca = '★ ELEGIDO' if k == k_optimo else ''
    print(f'  {k:>3}  {s:>12.4f}  {d:>10.4f}  {marca:>12}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Elbow ─────────────────────────────────────────────────────────────────
axes[0].plot(K_RANGE, inercias, marker='o', color=DS2,
             linewidth=2.5, markersize=8, zorder=3)
axes[0].fill_between(K_RANGE, inercias, alpha=0.08, color=DS3)
axes[0].axvline(k_optimo, color=DS5, linestyle='--', linewidth=2,
                label=f'K elegido = {k_optimo}')
axes[0].set_title('Método del Codo (Elbow)\nBuscar el "codo" de la curva',
                  fontweight='bold')
axes[0].set_xlabel('Número de clusters (K)')
axes[0].set_ylabel('Inercia')
axes[0].set_xticks(list(K_RANGE))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
axes[0].legend()

# ── Silhouette ────────────────────────────────────────────────────────────
colores_sil = [DS2 if k==k_optimo else DS4 for k in K_RANGE]
bars = axes[1].bar(K_RANGE, silhouettes, color=colores_sil,
                   edgecolor='white', width=0.6)
for bar, s in zip(bars, silhouettes):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{s:.3f}', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
axes[1].axhline(max(silhouettes), color=DS5, linestyle='--', linewidth=1.5,
                label=f'Máximo: {max(silhouettes):.3f} (K={k_optimo_sil})')
axes[1].axvline(k_optimo, color=DS2, linestyle='--', linewidth=1.5, alpha=0.7,
                label=f'K elegido = {k_optimo}')
axes[1].set_title('Silhouette Score por K\n(mayor = clusters más separados)',
                  fontweight='bold')
axes[1].set_xlabel('Número de clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(list(K_RANGE))
axes[1].legend(fontsize=8)

# ── Davies-Bouldin ────────────────────────────────────────────────────────
colores_db = [DS2 if k==k_optimo_db else DS4 for k in K_RANGE]
bars3 = axes[2].bar(K_RANGE, db_scores, color=colores_db,
                    edgecolor='white', width=0.6)
for bar, s in zip(bars3, db_scores):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{s:.3f}', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
axes[2].axhline(min(db_scores), color=DS5, linestyle='--', linewidth=1.5,
                label=f'Mínimo: {min(db_scores):.3f} (K={k_optimo_db})')
axes[2].set_title('Davies-Bouldin Index por K\n(menor = clusters más compactos)',
                  fontweight='bold')
axes[2].set_xlabel('Número de clusters (K)')
axes[2].set_ylabel('Davies-Bouldin Index')
axes[2].set_xticks(list(K_RANGE))
axes[2].legend(fontsize=8)

coincidencia = ('✅ Ambos coinciden' if k_optimo_sil == k_optimo_db
                else f'⚠️ Silhouette={k_optimo_sil}, DB={k_optimo_db}')
plt.suptitle(f'Selección del K óptimo → K = {k_optimo}  |  {coincidencia}',
             fontsize=13, fontweight='bold', color=DS1, y=1.01)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_08_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_08_elbow_silhouette.png')


In [ ]:
# Silhouette Diagram para el K óptimo
# Aquí ajustamos un KMeans temporal con el K elegido para diagnosticar la calidad
# antes de ejecutar el clustering definitivo de la siguiente sección.

km_final_temp = KMeans(n_clusters=k_optimo, random_state=42, n_init=20)
labels_diagram = km_final_temp.fit_predict(X_scaled)
sil_vals = silhouette_samples(X_scaled, labels_diagram)
sil_media = silhouette_score(X_scaled, labels_diagram)

fig, ax = plt.subplots(figsize=(10, max(6, k_optimo*1.5)))
y_lower = 10
colores_k = CLUSTER_PAL[:k_optimo]

for cid in range(k_optimo):
    sil_cid  = np.sort(sil_vals[labels_diagram == cid])
    size_cid = sil_cid.shape[0]
    y_upper  = y_lower + size_cid
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_cid,
                     facecolor=colores_k[cid], edgecolor=colores_k[cid], alpha=0.85)
    ax.text(-0.05, y_lower + size_cid/2, f'C{cid}  (n={size_cid:,})',
            ha='center', va='center', fontsize=9, fontweight='bold', color=colores_k[cid])
    y_lower = y_upper + 10

ax.axvline(sil_media, color=DS5, linestyle='--', linewidth=2,
           label=f'Media global = {sil_media:.3f}')
ax.axvline(0, color='gray', linestyle='-', linewidth=0.8, alpha=0.5)
ax.set_title(f'Silhouette Diagram — K={k_optimo}\n'
             f'Barras largas y anchas hacia la derecha = cluster bien definido\n'
             f'Barras que cruzan la línea roja = puntos mal asignados',
             fontweight='bold')
ax.set_xlabel('Silhouette coefficient (−1=mal asignado, 0=frontera, +1=bien asignado)')
ax.set_ylabel('Clientes (agrupados por cluster)')
ax.set_yticks([])
ax.legend(fontsize=10)
ax.set_xlim(-0.3, 1.0)

# Diagnóstico: puntos mal asignados
n_negativos = (sil_vals < 0).sum()
pct_neg = n_negativos / len(sil_vals) * 100
print(f'Clientes con silhouette < 0 (posible mal asignación): {n_negativos:,} ({pct_neg:.1f}%)')
print(f'→ Si >20% → considerar K diferente')

plt.tight_layout()
plt.savefig(GRAFICOS/'pca_09_silhouette_diagram.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_09_silhouette_diagram.png')

---
## 7. K-Means Clustering

Aplicamos K-Means con el K óptimo sobre el **espacio normalizado de features seleccionadas** (no sobre PCA). PCA solo se usa para visualizar los resultados.

In [ ]:
K_FINAL = k_optimo

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20)
df['cluster'] = km_final.fit_predict(X_scaled)

sil_final = silhouette_score(X_scaled, df['cluster'])
print(f'K-Means (K={K_FINAL}) completado.')
print(f'  Silhouette score: {sil_final:.4f}')
print(f'  Inercia final:    {km_final.inertia_:,.0f}')
print()

conteo = df['cluster'].value_counts().sort_index()
print('Clientes por cluster:')
for cid, n in conteo.items():
    bar = '█' * int(n/50)
    print(f'  C{cid}: {n:,} ({n/len(df)*100:.1f}%)  {bar}')

# Centroides en espacio original
centroides_scaled = km_final.cluster_centers_
centroides_orig   = scaler.inverse_transform(centroides_scaled)
centroides_df     = pd.DataFrame(centroides_orig, columns=FEATURES).round(4)
centroides_df['n_clientes'] = [conteo[c] for c in centroides_df.index]

# Proyección de centroides a PCA 2D
centroides_pca = pca2.transform(centroides_scaled)

print('\nCentroides (escala original):')
display(centroides_df)

In [ ]:
# Scatter PCA coloreado por cluster (visualización inicial)
paleta_k = CLUSTER_PAL[:K_FINAL]

fig, ax = plt.subplots(figsize=(11, 8))
for cid in sorted(df['cluster'].unique()):
    mask = df['cluster'] == cid
    ax.scatter(df.loc[mask,'pc1'], df.loc[mask,'pc2'],
               s=12, alpha=0.4, color=paleta_k[cid],
               label=f'Cluster {cid}  (n={mask.sum():,})')

ax.scatter(centroides_pca[:,0], centroides_pca[:,1],
           s=300, marker='X', color='black', zorder=5, linewidths=1.5, label='Centroide')
for cid,(cx,cy) in enumerate(centroides_pca):
    ax.text(cx+0.18, cy+0.18, f'C{cid}', fontweight='bold', fontsize=12)

ax.set_title(f'K-Means K={K_FINAL} — clusters en espacio PCA', fontweight='bold')
ax.set_xlabel(f'PC1 ({var1:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({var2:.1f}% varianza)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(GRAFICOS/'pca_10_scatter_clusters.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_10_scatter_clusters.png')

In [ ]:
# ── Análisis de estabilidad: 5 semillas distintas ─────────────────────────
# Verifica que el resultado no depende de la inicialización aleatoria

semillas = [42, 0, 7, 123, 999]
sil_estabilidad = []
inercia_estabilidad = []

print('Análisis de estabilidad del clustering:')
print(f'  {"Semilla":>8}  {"Silhouette":>12}  {"Inercia":>12}')
print('  ' + '-'*36)
for seed in semillas:
    km_s = KMeans(n_clusters=K_FINAL, random_state=seed, n_init=15)
    lbl_s = km_s.fit_predict(X_scaled)
    sil_s = silhouette_score(X_scaled, lbl_s)
    sil_estabilidad.append(sil_s)
    inercia_estabilidad.append(km_s.inertia_)
    print(f'  {seed:>8}  {sil_s:>12.4f}  {km_s.inertia_:>12,.0f}')

std_sil = np.std(sil_estabilidad)
rango_sil = max(sil_estabilidad) - min(sil_estabilidad)
print()
print(f'  Silhouette medio:   {np.mean(sil_estabilidad):.4f}')
print(f'  Desv. típica:       {std_sil:.4f}')
print(f'  Rango (max-min):    {rango_sil:.4f}')
print()
if std_sil < 0.005:
    print('✅ Clustering MUY ESTABLE — el resultado es robusto a la inicialización')
elif std_sil < 0.01:
    print('✅ Clustering ESTABLE — variación mínima entre semillas')
else:
    print('⚠️  Clustering INESTABLE — considerar aumentar n_init o cambiar K')

# Gráfico de estabilidad
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([str(s) for s in semillas], sil_estabilidad,
            color=DS3, edgecolor='white', width=0.5)
axes[0].axhline(np.mean(sil_estabilidad), color=DS5, linestyle='--',
                linewidth=1.5, label=f'Media: {np.mean(sil_estabilidad):.4f}')
for i, v in enumerate(sil_estabilidad):
    axes[0].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Silhouette score por semilla\n(estabilidad del clustering)',
                  fontweight='bold')
axes[0].set_xlabel('Semilla (random_state)')
axes[0].set_ylabel('Silhouette Score')
axes[0].legend()
axes[0].set_ylim(0, max(sil_estabilidad) * 1.15)

axes[1].bar([str(s) for s in semillas], inercia_estabilidad,
            color=DS2, edgecolor='white', width=0.5)
axes[1].axhline(np.mean(inercia_estabilidad), color=DS5, linestyle='--',
                linewidth=1.5, label=f'Media: {np.mean(inercia_estabilidad):,.0f}')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
axes[1].set_title('Inercia por semilla\n(consistencia de la solución)',
                  fontweight='bold')
axes[1].set_xlabel('Semilla (random_state)')
axes[1].set_ylabel('Inercia')
axes[1].legend()

plt.suptitle(f'Estabilidad del K-Means (K={K_FINAL}) — 5 inicializaciones distintas',
             fontsize=13, fontweight='bold', color=DS1, y=1.01)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_10b_estabilidad.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_10b_estabilidad.png')

---
## 8. Análisis profundo de clusters

Visualizamos el **perfil** de cada cluster para entender qué tipo de cliente representa cada grupo antes de ponerles nombre.

In [ ]:
# Métricas medias por cluster
perfil_medio = df.groupby('cluster')[FEATURES].mean()

# Normalización min-max por columna (para heatmap comparativo)
perfil_norm = (perfil_medio - perfil_medio.min()) / \
              (perfil_medio.max() - perfil_medio.min() + 1e-9)

# Labels dinámicos — solo las features que realmente están en FEATURES
LABELS_F_BASE = {
    'ingresos_t':      'Ingresos totales',
    'margen_t':        'Margen (%)',
    'frecuencia_t':    'Frecuencia (comp/mes)',
    'ticket_medio':    'Ticket medio (€)',
    'tasa_devolucion': 'Tasa devolución',
    'dias_sin_compra': 'Días sin compra',
}
LABELS_F = {k: v for k, v in LABELS_F_BASE.items() if k in FEATURES}

perfil_labeled = perfil_norm.rename(columns=LABELS_F)

fig, axes = plt.subplots(1, 2, figsize=(16, max(4, K_FINAL*0.9)))

# Heatmap normalizado
sns.heatmap(perfil_labeled.T, ax=axes[0],
            annot=True, fmt='.2f',
            cmap='YlGnBu', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Valor relativo (0=mínimo, 1=máximo)'})
axes[0].set_title('Perfil normalizado por cluster\n(0=mínimo, 1=máximo)', fontweight='bold')
axes[0].set_xlabel('Cluster')

# Heatmap valores reales — construir dinámicamente
perfil_real = df.groupby('cluster')[FEATURES].mean().rename(columns=LABELS_F)
perfil_fmt  = perfil_real.copy()

# Formatear cada columna según su tipo — solo si existe
if 'Ingresos totales'      in perfil_fmt.columns:
    perfil_fmt['Ingresos totales']      = perfil_fmt['Ingresos totales'].round(0)
if 'Margen (%)'            in perfil_fmt.columns:
    perfil_fmt['Margen (%)']            = (perfil_fmt['Margen (%)'] * 100).round(1)
if 'Frecuencia (comp/mes)' in perfil_fmt.columns:
    perfil_fmt['Frecuencia (comp/mes)'] = perfil_fmt['Frecuencia (comp/mes)'].round(3)
if 'Ticket medio (€)'      in perfil_fmt.columns:
    perfil_fmt['Ticket medio (€)']      = perfil_fmt['Ticket medio (€)'].round(0)
if 'Tasa devolución'       in perfil_fmt.columns:
    perfil_fmt['Tasa devolución']       = (perfil_fmt['Tasa devolución'] * 100).round(2)
if 'Días sin compra'       in perfil_fmt.columns:
    perfil_fmt['Días sin compra']       = perfil_fmt['Días sin compra'].round(0)

sns.heatmap(perfil_fmt.T, ax=axes[1],
            annot=True, fmt='.1f',
            cmap='Blues',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Valor real'})
axes[1].set_title('Valores reales por cluster\n(ingresos=€, margen/dev=%, frec=v/mes)',
                  fontweight='bold')
axes[1].set_xlabel('Cluster')

plt.suptitle('Perfil de cada cluster — normalizado y valores reales',
             fontsize=13, fontweight='bold', color=DS1, y=1.02)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_11_heatmap_clusters.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_11_heatmap_clusters.png')

In [ ]:
# Boxplots de las 4 métricas más importantes por cluster
feats_box  = ['cltv','ingresos_t','frecuencia_t','ticket_medio']
titulos_bx = ['CLTV (€)','Ingresos totales (€)','Frecuencia (comp/mes)','Ticket medio (€)']
clips_bx   = [df[f].quantile(0.95) for f in feats_box]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i,(feat,titulo,clip_v) in enumerate(zip(feats_box, titulos_bx, clips_bx)):
    data_k = [df.loc[df['cluster']==c, feat].clip(upper=clip_v).values
               for c in sorted(df['cluster'].unique())]
    bp = axes[i].boxplot(data_k,
                         labels=[f'C{c}' for c in sorted(df['cluster'].unique())],
                         patch_artist=True,
                         medianprops=dict(color='white', linewidth=2.5),
                         whiskerprops=dict(linewidth=1.2),
                         capprops=dict(linewidth=1.5))
    for patch, color in zip(bp['boxes'], paleta_k):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    axes[i].set_title(f'{titulo}\n(clip P95 = {clip_v:,.0f})', fontweight='bold')
    axes[i].set_xlabel('Cluster')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

plt.suptitle('Distribución de métricas clave por cluster',
             fontsize=13, fontweight='bold', color=DS1, y=1.01)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_12_boxplots.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_12_boxplots.png')

---
## 9. Radar chart por cluster

El **radar chart** (spider chart) es la visualización más clara para comparar múltiples clusters en varias dimensiones simultáneamente. Cada vértice es una feature; el área cubierta representa el "tamaño" de cada segmento.

In [ ]:
# Radar chart — recalcular angles dinámicamente según FEATURES actuales
LABELS_RADAR = [LABELS_F_BASE.get(f, f) for f in FEATURES]
N_AXES  = len(LABELS_RADAR)
angles  = [n / float(N_AXES) * 2 * np.pi for n in range(N_AXES)]
angles += angles[:1]

# Radar individual
fig, axes = plt.subplots(1, K_FINAL,
                          figsize=(4.5*K_FINAL, 5),
                          subplot_kw=dict(polar=True))
if K_FINAL == 1:
    axes = [axes]

for cid, ax in zip(range(K_FINAL), axes):
    valores  = perfil_norm.iloc[cid].values.tolist()
    valores += valores[:1]

    ax.plot(angles, valores, color=paleta_k[cid], linewidth=2.5)
    ax.fill(angles, valores, alpha=0.25, color=paleta_k[cid])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(LABELS_RADAR, fontsize=9)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=7, color='gray')
    ax.set_ylim(0, 1)
    n_cid = (df['cluster'] == cid).sum()
    ax.set_title(f'Cluster {cid}\n({n_cid:,} clientes)',
                 fontsize=10, pad=18, color=paleta_k[cid], fontweight='bold')

plt.suptitle('Radar Chart — Perfil individual de cada cluster (normalizado 0–1)',
             fontsize=13, fontweight='bold', color=DS1, y=1.03)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_13_radar_individual.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_13_radar_individual.png')

# Radar combinado
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for cid in range(K_FINAL):
    valores  = perfil_norm.iloc[cid].values.tolist()
    valores += valores[:1]
    n_cid = (df['cluster'] == cid).sum()
    ax.plot(angles, valores, color=paleta_k[cid], linewidth=2.5,
            label=f'Cluster {cid}  (n={n_cid:,})')
    ax.fill(angles, valores, alpha=0.08, color=paleta_k[cid])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(LABELS_RADAR, fontsize=11)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=8, color='gray')
ax.set_ylim(0, 1)
ax.set_title('Radar Chart combinado — todos los clusters\n(normalizado: 1=máximo, 0=mínimo)',
             fontsize=12, pad=20, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig(GRAFICOS/'pca_14_radar_combinado.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_13_radar_individual.png')
print(f'Guardado: {GRAFICOS}/pca_14_radar_combinado.png')

---
## 10. Interpretación y etiquetas de negocio

Con el perfil de cada cluster analizado, asignamos **nombres comprensibles** ordenados por valor económico (ingresos medios descendentes).

In [ ]:
perfil_completo = df.groupby('cluster').agg(
    n_clientes       = ('customer_id',     'count'),
    ingresos_medio   = ('ingresos_t',      'mean'),
    margen_medio_pct = ('margen_t',        'mean'),
    frecuencia_media = ('frecuencia_t',    'mean'),
    ticket_medio     = ('ticket_medio',    'mean'),
    tasa_dev_pct     = ('tasa_devolucion', 'mean'),
    cltv_medio       = ('cltv',            'mean'),
    ingresos_total   = ('ingresos_t',      'sum'),
).round(3)

perfil_completo['margen_medio_pct'] = (perfil_completo['margen_medio_pct']*100).round(1)
perfil_completo['tasa_dev_pct']     = (perfil_completo['tasa_dev_pct']*100).round(2)
perfil_completo['pct_clientes']     = (perfil_completo['n_clientes']/len(df)*100).round(1)
perfil_completo['pct_ingresos']     = (perfil_completo['ingresos_total']/df['ingresos_t'].sum()*100).round(1)

ranking = perfil_completo.sort_values('ingresos_medio', ascending=False).index.tolist()

print('=== PERFIL COMPLETO POR CLUSTER (orden por ingresos medios) ===')
display(perfil_completo.loc[ranking,
    ['n_clientes','pct_clientes','ingresos_medio','pct_ingresos',
     'margen_medio_pct','frecuencia_media','ticket_medio','tasa_dev_pct','cltv_medio']
].rename(columns={
    'n_clientes':'Clientes','pct_clientes':'% Cli',
    'ingresos_medio':'Ing. medio','pct_ingresos':'% Ing',
    'margen_medio_pct':'Margen %','frecuencia_media':'Frec.',
    'ticket_medio':'Ticket','tasa_dev_pct':'Dev %','cltv_medio':'CLTV medio'
}))

In [ ]:
NOMBRES_POR_K = {
    2: ['Alto valor',    'Bajo valor'],
    3: ['VIP',           'Regular',     'Bajo valor'],
    4: ['VIP',           'Alto valor',  'Regular',    'Bajo valor'],
    5: ['VIP',           'Alto valor',  'Regular',    'Bajo valor',  'Inactivos'],
    6: ['VIP',           'Alto valor',  'Regular',    'Ocasional',   'Bajo valor',  'Inactivos'],
    7: ['VIP',           'Fidelizados', 'Alto valor',  'Regular',    'Ocasional',   'Bajo valor', 'Inactivos'],
    8: ['VIP',           'Fidelizados', 'Alto valor',  'Regular',    'Ocasional',   'Bajo valor', 'En riesgo', 'Inactivos'],
    9: ['VIP',           'Fidelizados', 'Alto valor',  'Regular',    'Ocasional',   'Bajo valor', 'En riesgo', 'Potenciales', 'Inactivos'],
}
nombres_lista = NOMBRES_POR_K.get(K_FINAL, [f'Grupo {i+1}' for i in range(K_FINAL)])

# ranking ya está ordenado por ingresos_medio descendente
nombres_cluster = {cid: nombre for cid, nombre in zip(ranking, nombres_lista)}
df['cluster_label'] = df['cluster'].map(nombres_cluster)
color_label = {nombres_cluster[cid]: CLUSTER_PAL[cid] for cid in range(K_FINAL)}

# ── Validación de coherencia de etiquetas ────────────────────────────────
# Comprobamos que las propiedades de cada cluster son coherentes con su nombre
print('=== VALIDACIÓN DE ETIQUETAS ===')
print()

alertas = []
for i, cid in enumerate(ranking):
    label   = nombres_cluster[cid]
    ing     = perfil_completo.loc[cid, 'ingresos_medio']
    frec    = perfil_completo.loc[cid, 'frecuencia_media']
    dev     = perfil_completo.loc[cid, 'tasa_dev_pct']
    cltv_m  = perfil_completo.loc[cid, 'cltv_medio']
    n       = perfil_completo.loc[cid, 'n_clientes']

    # Verificar que el orden de ingresos es decreciente
    if i > 0:
        ing_prev = perfil_completo.loc[ranking[i-1], 'ingresos_medio']
        if ing > ing_prev:
            alertas.append(f'⚠️  {label} tiene más ingresos que {nombres_cluster[ranking[i-1]]} — revisar orden')

    # Verificar que "Inactivos" tiene alta tasa de dias_sin_compra si existe
    if label == 'Inactivos' and 'dias_sin_compra' in df.columns:
        dias_med = df.loc[df['cluster'] == cid, 'dias_sin_compra'].mean()
        if dias_med < 180:
            alertas.append(f'⚠️  Cluster "{label}" tiene media {dias_med:.0f} días sin compra '
                          f'— puede no ser realmente inactivo (umbral=180)')

    # Verificar que "VIP" tiene frecuencia alta
    if label == 'VIP':
        frec_global = df['frecuencia_t'].mean()
        if frec < frec_global:
            alertas.append(f'⚠️  Cluster "VIP" tiene frecuencia {frec:.3f} < media global {frec_global:.3f}')

    print(f'  Cluster {cid} → "{label}"')
    print(f'    Ingresos medio: {ing:>10,.0f} €  |  Frecuencia: {frec:.3f} v/mes  '
          f'|  Dev: {dev:.1f}%  |  CLTV: {cltv_m:,.0f} €  |  n={n:,}')

print()
if alertas:
    print('ALERTAS DETECTADAS:')
    for a in alertas:
        print(f'  {a}')
    print()
    print('→ Revisar manualmente las etiquetas en NOMBRES_POR_K y ajustar si es necesario.')
else:
    print('✅ Todas las etiquetas son coherentes con el perfil de cada cluster.')

In [ ]:
# Scatter PCA final — versión mejorada con densidad y separación visual
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Izquierda: scatter con contornos de densidad ──────────────────────────
ax = axes[0]
for cid in ranking:
    mask  = df['cluster'] == cid
    label = nombres_cluster[cid]
    color = CLUSTER_PAL[cid]
    x_c   = df.loc[mask, 'pc1']
    y_c   = df.loc[mask, 'pc2']

    # Puntos
    ax.scatter(x_c, y_c, s=14, alpha=0.35, color=color, zorder=2)

    # Elipse de confianza (1 sigma) alrededor del cluster
    from matplotlib.patches import Ellipse
    cov  = np.cov(x_c, y_c)
    vals, vecs = np.linalg.eigh(cov)
    orden = vals.argsort()[::-1]
    vals, vecs = vals[orden], vecs[:, orden]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    ancho, alto = 2 * np.sqrt(vals)
    elipse = Ellipse(xy=(x_c.mean(), y_c.mean()),
                     width=ancho, height=alto, angle=theta,
                     edgecolor=color, facecolor=color,
                     alpha=0.12, linewidth=2, zorder=1)
    ax.add_patch(elipse)
    elipse2 = Ellipse(xy=(x_c.mean(), y_c.mean()),
                      width=ancho, height=alto, angle=theta,
                      edgecolor=color, facecolor='none',
                      alpha=0.7, linewidth=1.5, linestyle='--', zorder=3)
    ax.add_patch(elipse2)

# Centroides
ax.scatter(centroides_pca[:, 0], centroides_pca[:, 1],
           s=350, marker='X', color='black', zorder=6,
           linewidths=1.5, label='Centroides')
for cid, (cx, cy) in enumerate(centroides_pca):
    ax.text(cx + 0.15, cy + 0.15, nombres_cluster[cid],
            fontsize=10, fontweight='bold', color=CLUSTER_PAL[cid],
            zorder=7)

ax.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
ax.axvline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
ax.set_title(f'K-Means K={K_FINAL} con elipses de confianza (1σ)\n'
             f'Silhouette={sil_final:.3f}  |  Varianza PCA={var1+var2:.1f}%',
             fontweight='bold')
ax.set_xlabel(f'PC1 ({var1:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({var2:.1f}% varianza)')

# Leyenda manual
handles = [mpatches.Patch(color=CLUSTER_PAL[cid],
           label=f'{nombres_cluster[cid]}  (n={int(perfil_completo.loc[cid,"n_clientes"]):,})')
           for cid in ranking]
ax.legend(handles=handles, fontsize=9, title='Segmento', title_fontsize=9)

# ── Derecha: scatter coloreado por CLTV continuo ─────────────────────────
ax2 = axes[1]
clip_cltv = df['cltv'].quantile(0.97)
sc = ax2.scatter(df['pc1'], df['pc2'],
                 c=df['cltv'].clip(upper=clip_cltv),
                 cmap='YlOrRd', s=10, alpha=0.5, zorder=2)
plt.colorbar(sc, ax=ax2, label=f'CLTV (€, clip P97={clip_cltv:,.0f})')
ax2.scatter(centroides_pca[:, 0], centroides_pca[:, 1],
            s=300, marker='X', color='black', zorder=5, linewidths=1.5)
for cid, (cx, cy) in enumerate(centroides_pca):
    ax2.text(cx + 0.15, cy + 0.15, nombres_cluster[cid],
             fontsize=9, fontweight='bold', color='black', zorder=6)
ax2.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
ax2.axvline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
ax2.set_title('Distribución de CLTV en el espacio PCA\n'
              '(amarillo=bajo, rojo=alto)',
              fontweight='bold')
ax2.set_xlabel(f'PC1 ({var1:.1f}% varianza)')
ax2.set_ylabel(f'PC2 ({var2:.1f}% varianza)')

plt.suptitle(f'Segmentación de clientes — K-Means K={K_FINAL}',
             fontsize=14, fontweight='bold', color=DS1, y=1.01)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_15_scatter_final.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_15_scatter_final.png')

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Scatter interactivo con hover informativo
df_plot = df.copy()
df_plot['cltv_fmt']      = df_plot['cltv'].round(0).astype(int)
df_plot['ingresos_fmt']  = df_plot['ingresos_t'].round(0).astype(int)
df_plot['cluster_nombre'] = df_plot['cluster_label']

fig_px = px.scatter(
    df_plot,
    x='pc1', y='pc2',
    color='cluster_nombre',
    color_discrete_sequence=CLUSTER_PAL[:K_FINAL],
    hover_data={
        'full_name':      True,
        'cltv_fmt':       True,
        'ingresos_fmt':   True,
        'frecuencia_t':   ':.3f',
        'pc1':            False,
        'pc2':            False,
        'cluster_nombre': False,
    },
    labels={
        'pc1':            f'PC1 ({var1:.1f}% varianza)',
        'pc2':            f'PC2 ({var2:.1f}% varianza)',
        'cluster_nombre': 'Segmento',
        'cltv_fmt':       'CLTV (€)',
        'ingresos_fmt':   'Ingresos (€)',
        'full_name':      'Cliente',
    },
    title=f'Segmentación de clientes — K-Means K={K_FINAL}'
          f'<br><sup>Silhouette={sil_final:.3f}  |  Varianza PCA={var1+var2:.1f}%</sup>',
    opacity=0.55,
    width=1000, height=680,
)

# Añadir centroides como marcadores X negros
fig_px.add_trace(go.Scatter(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    mode='markers+text',
    marker=dict(symbol='x', size=16, color='black', line=dict(width=3)),
    text=[nombres_cluster[cid] for cid in range(K_FINAL)],
    textposition='top right',
    textfont=dict(size=11, color='black'),
    name='Centroides',
    showlegend=True,
))

fig_px.update_traces(marker=dict(size=6), selector=dict(mode='markers'))
fig_px.update_layout(
    plot_bgcolor='#F8FAFC',
    paper_bgcolor='white',
    title_font=dict(size=14, color=DS1),
    legend_title_text='Segmento',
    font=dict(family='DejaVu Sans'),
)
fig_px.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4)
fig_px.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.4)

fig_px.show()
fig_px.write_html(str(GRAFICOS / 'pca_scatter_interactivo.html'))
print('Guardado: graficos/pca_scatter_interactivo.html')

In [ ]:
print('=== TU SEGMENTACIÓN ACTUAL ===')
print()
print(f'K óptimo encontrado: {K_FINAL}')
print()
for cid in ranking:
    label = nombres_cluster[cid]
    n     = int(perfil_completo.loc[cid, 'n_clientes'])
    ing   = perfil_completo.loc[cid, 'ingresos_medio']
    cltv  = perfil_completo.loc[cid, 'cltv_medio']
    frec  = perfil_completo.loc[cid, 'frecuencia_media']
    pct_i = perfil_completo.loc[cid, 'pct_ingresos']
    print(f'  {label:<15} → {n:>5,} clientes ({n/len(df)*100:.1f}%)  '
          f'| Ing.medio={ing:>8,.0f}€  '
          f'| CLTV={cltv:>8,.0f}€  '
          f'| {pct_i:.1f}% ingresos')

In [ ]:
# Barras: clientes e ingresos por segmento
etiquetas_ord = nombres_lista  # orden por valor
resumen_bar = df.groupby('cluster_label').agg(
    n_clientes     = ('customer_id','count'),
    ingresos_total = ('ingresos_t','sum')
)
# Ordenar según ranking
resumen_bar = resumen_bar.reindex([nombres_cluster[c] for c in ranking])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colores_ord = [CLUSTER_PAL[cid] for cid in ranking]

# Clientes
bars1 = axes[0].bar(etiquetas_ord, resumen_bar['n_clientes'],
                    color=colores_ord, edgecolor='white', width=0.6)
for bar,v in zip(bars1, resumen_bar['n_clientes']):
    pct = v/len(df)*100
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 f'{v:,}\n({pct:.1f}%)', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Clientes por segmento', fontweight='bold')
axes[0].set_ylabel('Número de clientes')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylim(0, resumen_bar['n_clientes'].max()*1.25)

# Ingresos
ing_M = resumen_bar['ingresos_total']/1e6
bars2 = axes[1].bar(etiquetas_ord, ing_M,
                    color=colores_ord, edgecolor='white', width=0.6)
for bar,v in zip(bars2, resumen_bar['ingresos_total']):
    pct = v/df['ingresos_t'].sum()*100
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()/1e6+0.05,
                 f'{v/1e6:.2f}M€\n({pct:.1f}%)', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Ingresos totales por segmento', fontweight='bold')
axes[1].set_ylabel('Ingresos (millones €)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylim(0, ing_M.max()*1.25)

plt.suptitle('Distribución de clientes e ingresos por segmento',
             fontsize=13, fontweight='bold', color=DS1, y=1.01)
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_16_barras_segmentos.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_16_barras_segmentos.png')

---
## 11. Comparativa CLTV vs Cluster

¿Coincide la segmentación automática (K-Means) con la segmentación manual (CLTV percentiles)? Esta tabla cruzada revela si el clustering ha capturado la misma estructura o ha aportado información adicional.

In [ ]:
# Tabla cruzada: segmento CLTV vs cluster label
if 'segmento_cltv' in df.columns:
    tabla_cruzada = pd.crosstab(
        df['segmento_cltv'], df['cluster_label'],
        margins=True, margins_name='TOTAL'
    ).reindex(index=['Alto','Medio','Bajo','TOTAL'])

    print('Tabla cruzada: Segmento CLTV × Cluster K-Means')
    display(tabla_cruzada)

    # Heatmap de la tabla cruzada (sin TOTAL)
    tabla_pct = tabla_cruzada.iloc[:-1, :-1].div(
        tabla_cruzada.iloc[:-1, :-1].sum(axis=1), axis=0
    ) * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Heatmap conteos
    sns.heatmap(tabla_cruzada.iloc[:-1,:-1], ax=axes[0],
                annot=True, fmt='d', cmap='YlGnBu',
                linewidths=0.5, linecolor='white',
                cbar_kws={'label':'Nº clientes'})
    axes[0].set_title('Conteo: Segmento CLTV × Cluster', fontweight='bold')
    axes[0].set_xlabel('Cluster K-Means')
    axes[0].set_ylabel('Segmento CLTV')

    # Heatmap porcentajes (por fila)
    sns.heatmap(tabla_pct, ax=axes[1],
                annot=True, fmt='.1f', cmap='YlOrRd',
                linewidths=0.5, linecolor='white',
                cbar_kws={'label':'% dentro del segmento CLTV'})
    axes[1].set_title('Distribución % por segmento CLTV → clusters\n(% por fila)', fontweight='bold')
    axes[1].set_xlabel('Cluster K-Means')
    axes[1].set_ylabel('Segmento CLTV')

    plt.suptitle('Validación cruzada: CLTV vs K-Means',
                 fontsize=13, fontweight='bold', color=DS1, y=1.01)
    plt.tight_layout()
    plt.savefig(GRAFICOS/'pca_17_comparativa_cltv.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Guardado: {GRAFICOS}/pca_17_comparativa_cltv.png')

    print()
    print('Interpretación:')
    print('  Si los clientes CLTV-Alto se concentran en 1-2 clusters → K-Means es consistente con el CLTV')
    print('  Si están dispersos → K-Means ha detectado sub-grupos dentro de cada segmento CLTV')
else:
    print('Columna segmento_cltv no disponible.')

In [ ]:
# Adjusted Rand Index — concordancia cuantitativa entre CLTV y K-Means
from sklearn.metrics import adjusted_rand_score

seg_map = {'Alto': 2, 'Medio': 1, 'Bajo': 0}
ari = adjusted_rand_score(
    df['segmento_cltv'].map(seg_map),
    df['cluster']
)

print(f'Adjusted Rand Index (CLTV percentiles vs K-Means): {ari:.3f}')
print()
if ari > 0.6:
    interpretacion = ('Alta concordancia — K-Means reproduce esencialmente '
                      'la misma segmentación que el CLTV por percentiles.')
elif ari > 0.3:
    interpretacion = ('Concordancia moderada — K-Means confirma la estructura '
                      'del CLTV pero detecta sub-grupos adicionales dentro de '
                      'cada segmento, aportando granularidad extra.')
elif ari > 0.0:
    interpretacion = ('Baja concordancia — K-Means ha encontrado una estructura '
                      'diferente a la del CLTV. El clustering aporta información '
                      'nueva no capturada por los percentiles.')
else:
    interpretacion = ('ARI negativo o nulo — el clustering es independiente '
                      'del CLTV. Ambas segmentaciones capturan dimensiones '
                      'distintas del comportamiento del cliente.')

print(f'Interpretación: {interpretacion}')
print()
print('Referencia ARI:')
print('  ARI = 1.0 → segmentaciones idénticas')
print('  ARI = 0.0 → independientes (esperado si K-Means aporta valor adicional)')
print('  ARI < 0   → peor que aleatorio (muy raro en datos reales)')

# Mini gráfico del ARI como gauge
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.barh(0, ari, color=DS2 if ari > 0.3 else DS5,
        edgecolor='white', height=0.4)
ax.barh(0, 1.0, color=DS4, edgecolor='white', height=0.4, alpha=0.3)
ax.axvline(ari, color=DS1, linewidth=2)
ax.text(ari + 0.02, 0, f'ARI = {ari:.3f}', va='center',
        fontsize=13, fontweight='bold', color=DS1)
ax.set_xlim(min(-0.1, ari - 0.1), 1.1)
ax.set_yticks([])
ax.set_xlabel('Adjusted Rand Index (0 = independiente, 1 = idéntico)')
ax.set_title('Concordancia entre segmentación CLTV y K-Means',
             fontweight='bold')
plt.tight_layout()
plt.savefig(GRAFICOS/'pca_17b_ari.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Guardado: {GRAFICOS}/pca_17b_ari.png')

---
## 12. Exportación y KPIs finales

In [ ]:
# Columnas base del CSV de salida
cols_base = ['customer_id','ingresos_t','margen_t','frecuencia_t',
             'ticket_medio','tasa_devolucion','cltv','pc1','pc2',
             'cluster','cluster_label']

# Añadir columnas extra si existen
for col in ['full_name','segmento_cltv','dias_sin_compra','churn_proxy',
            'R_score','F_score','M_score','segmento_rfm']:
    if col in df.columns:
        cols_base.insert(2, col) if col == 'full_name' else cols_base.append(col)

# Deduplicar conservando orden
seen = set()
cols_export = [c for c in cols_base if not (c in seen or seen.add(c))]
cols_export = [c for c in cols_export if c in df.columns]

df[cols_export].to_csv('clustering_resultados.csv', index=False)

# Verificaciones
df_check = pd.read_csv('clustering_resultados.csv')
n_esperado = len(df)
assert len(df_check) == n_esperado, f'Esperado {n_esperado}, obtenido {len(df_check)}'
assert df_check['customer_id'].nunique() == n_esperado, 'customer_id no es único'
assert df_check['cluster_label'].notna().all(), 'Hay NaN en cluster_label'

print('clustering_resultados.csv exportado y verificado.')
print(f'  Filas:    {len(df_check):,}')
print(f'  Columnas: {len(df_check.columns)}')
print(f'  Segmentos: {df_check["cluster_label"].unique().tolist()}')

In [ ]:
# Panel KPIs finales — versión compatible VSCode/Jupyter
seg_top_label = nombres_cluster[ranking[0]]
seg_top_n     = resumen_bar.loc[seg_top_label, 'n_clientes']
seg_top_pct   = seg_top_n / len(df) * 100
seg_top_ing   = resumen_bar.loc[seg_top_label, 'ingresos_total'] / df['ingresos_t'].sum() * 100

kpis = [
    (f'{len(df):,}',                          'Clientes\nanalizados',                     DS1),
    (f'K = {K_FINAL}',                        'Clusters\nóptimos',                        DS2),
    (f'{sil_final:.3f}',                      'Silhouette score\n(0=peor, 1=mejor)',       DS3),
    (f'{var1+var2:.0f}%',                     'Varianza capturada\nPCA 2D',               '#00897B'),
    (f'{seg_top_n:,}\n({seg_top_pct:.0f}%)',  f'Clientes\n"{seg_top_label}"',             DS5),
    (f'{seg_top_ing:.1f}%',                   f'Ingresos del\nsegmento "{seg_top_label}"', DS6),
]

fig, axes = plt.subplots(1, len(kpis), figsize=(20, 3.5))
fig.patch.set_facecolor('white')

for ax, (valor, etiq, color) in zip(axes, kpis):
    # Fondo: rectángulo que ocupa todo el axis
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    rect = plt.Rectangle((0, 0), 1, 1,
                          transform=ax.transAxes,
                          facecolor=color,
                          edgecolor='white',
                          linewidth=3)
    ax.add_patch(rect)
    # Valor principal
    ax.text(0.5, 0.62, valor,
            transform=ax.transAxes,
            ha='center', va='center',
            fontsize=17, fontweight='bold', color='white',
            linespacing=1.2)
    # Etiqueta
    ax.text(0.5, 0.22, etiq,
            transform=ax.transAxes,
            ha='center', va='center',
            fontsize=9, color='white', alpha=0.92,
            linespacing=1.3)
    # Ocultar ejes pero mantener el fondo
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle('Resumen ejecutivo — PCA + Clustering (Fase 6)',
             fontsize=13, fontweight='bold', color=DS1, y=1.04)

plt.tight_layout(pad=0.3)
plt.savefig(GRAFICOS/'pca_18_kpi_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'Guardado: {GRAFICOS}/pca_18_kpi_dashboard.png')

In [ ]:
# ── Timestamp y resumen de ejecución ─────────────────────────────────────
import datetime

ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print('=' * 60)
print(f'Fase 6 completada: {ts}')
print('=' * 60)
print(f'  Clientes procesados:     {len(df):,}')
print(f'  Features utilizadas:     {len(FEATURES)} → {FEATURES}')
print(f'  K óptimo:                {K_FINAL}')
print(f'  Silhouette final:        {sil_final:.4f}')
print(f'  Davies-Bouldin final:    {db_scores[list(K_RANGE).index(K_FINAL)]:.4f}')
print(f'  Varianza PCA 2D:         {var1+var2:.1f}%')
print(f'  Segmento top:            "{nombres_cluster[ranking[0]]}" '
      f'({perfil_completo.loc[ranking[0],"n_clientes"]:,} clientes, '
      f'{perfil_completo.loc[ranking[0],"pct_ingresos"]:.1f}% ingresos)')
print(f'  Output:                  clustering_resultados.csv')
print(f'  Gráficos generados:      graficos/pca_*.png')
print('=' * 60)
print()
print('Siguientes fases:')
print('  → Fase 7: 07_dashboard.py  (Streamlit)')
print('  → Fase 8: 06_documento_tecnico.md')

---
## Conclusiones de la Fase 6

### Decisiones metodológicas tomadas

| Decisión | Justificación |
|---|---|
| Features seleccionadas (set final) | Representativas de dimensiones de negocio distintas, evitando colapso en una sola variable |
| Outliers: IQR×3 clip | Conservador — nunca eliminar clientes (política del proyecto) |
| StandardScaler | Escalas muy distintas (€ vs ratios 0-1) |
| K-Means en espacio normalizado | PCA solo para visualizar; clustering en el espacio completo de features efectivas |
| K óptimo por Silhouette | Más robusto que solo el codo, confirmado visualmente con Silhouette Diagram |

### Outputs generados

| Archivo | Descripción |
|---|---|
| `clustering_resultados.csv` | 5.750 clientes con `cluster` + `cluster_label` asignados |
| `graficos/pca_*.png` | 19 gráficos de esta fase |

### Próximas fases
- **Fase 7** (`07_dashboard.py`): dashboard Streamlit con visualización interactiva
- **Fase 8** (`06_documento_tecnico.md`): memoria técnica final